# LeetCode 2729 — Check if The Number is Fascinating bằng Python

Link đề: https://leetcode.com/problems/check-if-the-number-is-fascinating/ · Độ khó: **Easy** · Chủ đề: Hash Table, Math

Skeleton LeetCode:

```python
class Solution:
    def isFascinating(self, n: int) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ghép (**nối chuỗi**, không phải cộng) `n`, `2·n`, `3·n` thành một chuỗi; chuỗi đó "kỳ diệu" (fascinating) nếu chứa **mỗi chữ số từ `1` đến `9` đúng một lần** và **không có chữ số `0`**.
2. Phân biệt **nối chuỗi** (`"192" + "384"`) và **cộng số** (`192 + 384`).
3. Dùng `str(n)`, `sorted`, `set`, `len`, `count`, và cách tách chữ số bằng `% 10`, `// 10`.
4. Viết 4 cách: đếm từng chữ số `1..9` (brute), sắp xếp rồi so với `"123456789"`, `set` + độ dài, và tách chữ số bằng số học với mảng đánh dấu (thoát sớm).
5. Phát hiện bằng vét cạn toàn miền `100..999` rằng chỉ có **4 số** thoả; vì sao ta **không nên** dùng bảng tra cứng như lời giải chính.
6. Tránh các lỗi quen thuộc: cộng số thay vì nối chuỗi, chỉ kiểm tra `set` mà quên độ dài, quên loại chữ số `0`, so sánh `sorted(...)` (list) với chuỗi.

## 2. Đề bài & ví dụ

Cho một số nguyên `n` gồm **đúng 3 chữ số**. Ta gọi `n` là **fascinating** nếu sau khi **nối** (concatenate) `n` với `2 * n` và `3 * n`, chuỗi kết quả thoả cả hai điều kiện:

- chứa **tất cả** các chữ số từ `1` đến `9` **đúng một lần**;
- **không chứa** chữ số `0`.

Trả về `True` nếu `n` là fascinating, ngược lại `False`.

"Nối" nghĩa là viết liền nhau: nối `121` với `371` được `121371`.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `192` | `True` | `192`, `384`, `576` nối thành `192384576`; có đủ `1..9` mỗi chữ số đúng một lần, không có `0` |
| 2 | `100` | `False` | `100`, `200`, `300` nối thành `100200300`; có `0` và lặp chữ số |

Constraints: `100 <= n <= 999`.

In [1]:
n = 192
s = str(n) + str(2 * n) + str(3 * n)
print(s, len(s))                         # 192384576 9
print(sorted(s))                          # ['1', '2', ..., '9']
n = 100
print(str(n) + str(2 * n) + str(3 * n))   # 100200300

192384576 9
['1', '2', '3', '4', '5', '6', '7', '8', '9']
100200300


## 3. Kiến thức nền

### 3.1. `str(n)`, nối chuỗi `+` và nhân số

`str(192)` là `"192"`. Dấu `+` giữa **hai chuỗi** là **nối** (`"192" + "384" = "192384"`), còn `+` giữa **hai số** là **cộng** (`192 + 384 = 576`). Dễ nhầm — phải đổi sang chuỗi **trước** khi nối.

In [2]:
print("192" + "384")                 # 192384  (nối)
print(192 + 384)                     # 576     (cộng)
n = 192
print(str(n) + str(2 * n) + str(3 * n))     # 192384576
print(type(str(n)), type(n))

192384
576
192384576
<class 'str'> <class 'int'>


### 3.2. `sorted`, `set`, `len`, `count`

- `sorted(s)` trả về **list** các ký tự đã sắp tăng dần (không phải chuỗi!).
- `"".join(sorted(s))` ghép lại thành chuỗi đã sắp xếp.
- `set(s)` là tập các ký tự **khác nhau** (bỏ trùng).
- `len(s)` độ dài; `s.count("1")` đếm số lần `"1"` xuất hiện.

In [3]:
s = "192384576"
print(sorted(s))                       # ['1', '2', '3', '4', '5', '6', '7', '8', '9']
print("".join(sorted(s)))              # 123456789
print(sorted(s) == "123456789")        # False! list != str
print("".join(sorted(s)) == "123456789")   # True
print(set(s), len(set(s)), len(s))     # 9 chữ số khác nhau, độ dài 9
print(s.count("1"), s.count("0"))      # 1 0

['1', '2', '3', '4', '5', '6', '7', '8', '9']
123456789
False
True
{'7', '9', '4', '1', '2', '3', '8', '5', '6'} 9 9
1 0


### 3.3. Tách chữ số bằng số học: `% 10` và `// 10`

Không cần đổi sang chuỗi: `x % 10` là chữ số cuối, `x //= 10` bỏ chữ số cuối. Lặp tới khi `x == 0` ta duyệt hết mọi chữ số (từ phải sang trái). Có thể gói gọn bằng `divmod(x, 10)` cho cả thương và dư.

In [4]:
x = 576
digits = []
while x > 0:
    digits.append(x % 10)
    x //= 10
print(digits)                 # [6, 7, 5]
print(divmod(576, 10))        # (57, 6)

[6, 7, 5]
(57, 6)


### 3.4. Mảng đếm / mảng đánh dấu 10 ô cho chữ số `0..9`

`seen = [0] * 10` tạo 10 ô; `seen[d] += 1` đếm số lần chữ số `d` xuất hiện. Sau khi đếm, điều kiện fascinating là: `seen[0] == 0` và `seen[d] == 1` với mọi `d` từ `1` đến `9`. Hàm `all(...)` kiểm tra mọi điều kiện trong một dòng.

In [5]:
cnt = [0] * 10
for ch in "192384576":
    cnt[int(ch)] += 1
print(cnt)                                          # [0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
print(cnt[0] == 0 and all(cnt[d] == 1 for d in range(1, 10)))   # True
print(all([]), all([True, False]))                  # True False

[0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
True
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ta chỉ cần làm đúng như đề:

1. Tạo chuỗi `s = str(n) + str(2n) + str(3n)`.
2. Kiểm tra `s` **có đúng 9 chữ số**, trong đó mỗi chữ số `1..9` **xuất hiện đúng một lần** (suy ra không có `0`, vì 9 vị trí đã dùng hết cho `1..9`).

Tính tay `n = 192`: `2n = 384`, `3n = 576` ⇒ `s = "192384576"`. Đếm: `1,2,3,4,5,6,7,8,9` mỗi chữ số đúng một lần ✓.

Tính tay `n = 100`: `s = "100200300"` có `0` ⇒ ✗.

**Nhận xét quan trọng:** vì phải có đúng 9 chữ số, và `n` đã có 3 chữ số, mà `2n` và `3n` mỗi cái ≥ 3 chữ số, nên `2n` và `3n` đều phải đúng 3 chữ số ⇒ `3n ≤ 999` ⇒ `n ≤ 333`. Đó là lý do độ dài `s` là một điều kiện **cần**.

In [6]:
for n in (192, 100, 267, 500):
    s = str(n) + str(2 * n) + str(3 * n)
    print(f"n={n}: s={s} (độ dài {len(s)}), có 0? {'0' in s}, các chữ khác nhau: {len(set(s))}")

n=192: s=192384576 (độ dài 9), có 0? False, các chữ khác nhau: 9
n=100: s=100200300 (độ dài 9), có 0? True, các chữ khác nhau: 4
n=267: s=267534801 (độ dài 9), có 0? True, các chữ khác nhau: 9
n=500: s=50010001500 (độ dài 11), có 0? True, các chữ khác nhau: 3


## 5. Cách 1 — Brute Force / baseline: đếm từng chữ số `1..9`

**Ý tưởng:** dịch **nguyên văn** điều kiện của đề thành code: (1) `"0"` không có trong `s`; (2) với mỗi `d` từ `1` đến `9`: `s.count(str(d)) == 1`. Nếu cả hai đúng ⇒ `True`. Đề không nói "độ dài 9" nhưng điều kiện (1) và (2) kéo theo độ dài `9` (mỗi chữ số `1..9` đúng một lần, không có `0`, và `s` chỉ gồm chữ số nên không còn ký tự nào khác — chi tiết ở mục 10).

In [7]:
def fascinating_count(n: int) -> bool:
    s = str(n) + str(2 * n) + str(3 * n)
    if "0" in s:
        return False
    for d in range(1, 10):
        if s.count(str(d)) != 1:
            return False
    return True

print(fascinating_count(192), fascinating_count(100), fascinating_count(267))     # True False False

True False False


**Độ phức tạp:** `s` dài tối đa 11 ký tự (`3 + 4 + 4`); mỗi `count` quét `s` ⇒ 9 lần quét `O(|s|)`. Gọi `L = |s| ≤ 11`: Time `O(9·L)` = **hằng số** vì `n` chỉ có 3 chữ số. Space `O(L)`.

**Nút thắt:** không đáng kể ở đề này (mọi cách đều `O(1)` theo cỡ đầu vào). Nhưng 9 lần `count` hơi rườm rà — có thể gộp bằng sắp xếp hoặc set.

## 6. Ý tưởng thuật toán / bước đột phá

"Mỗi chữ số `1..9` đúng một lần" nghĩa là **nếu ta sắp xếp các ký tự của `s` thì được đúng `"123456789"`**. Một phép so sánh duy nhất thay cho 9 lần đếm!

Lập luận: chuỗi đã sắp xếp bằng `"123456789"` ⇔ `s` là một **hoán vị** của các ký tự `1,2,...,9` ⇔ mỗi chữ số `1..9` xuất hiện đúng một lần, độ dài 9, không có `0`. Chính là điều kiện của đề.

Một cách khác: `set(s)` chứa đúng 9 chữ số khác nhau `1..9` **và** `len(s) == 9` (không có chữ số nào lặp).

In [8]:
s = "192384576"
print("".join(sorted(s)) == "123456789")             # True
s2 = "100200300"
print("".join(sorted(s2)), "".join(sorted(s2)) == "123456789")   # 000000123 False

True
000000123 False


## 7. Các cách tiếp theo

Ngoài Cách 1 (đếm từng chữ số), ta xem 3 cách: (2) sắp xếp rồi so với `"123456789"`, (3) `set` + độ dài, (4) tách chữ số bằng số học với mảng đếm và **thoát sớm**.

### Cách 2 — Sắp xếp rồi so sánh

Giải thích từng đoạn:

- `s = str(n) + str(2 * n) + str(3 * n)`: nối ba chuỗi.
- `sorted(s)`: list các ký tự đã sắp tăng dần.
- `"".join(...)`: ghép lại thành chuỗi.
- `== "123456789"`: so khớp đúng một hoán vị của các chữ số `1..9`.

In [9]:
def fascinating_sorted(n: int) -> bool:
    s = str(n) + str(2 * n) + str(3 * n)
    return "".join(sorted(s)) == "123456789"

print(fascinating_sorted(192), fascinating_sorted(100), fascinating_sorted(267))

True False False


In [10]:
class SolutionSorted:
    def isFascinating(self, n: int) -> bool:
        s = str(n) + str(2 * n) + str(3 * n)
        return "".join(sorted(s)) == "123456789"

print(SolutionSorted().isFascinating(192))

True


**Độ phức tạp:** sắp xếp `L ≤ 11` ký tự: `O(L log L)` — hằng số vì `L ≤ 11`. Space `O(L)`.

### Cách 3 — `set` + độ dài

Giải thích: `len(s) == 9` đảm bảo đúng 9 chữ số; `set(s) == set("123456789")` đảm bảo tập chữ số là `{1,...,9}` (suy ra không có `0`). Hai điều kiện cùng lúc ⇒ 9 ký tự lấy từ tập 9 phần tử mà phủ kín cả tập ⇒ mỗi chữ số đúng một lần (nguyên lý chuồng bồ câu).

**Cả hai điều kiện đều cần:** chỉ có `set` thì bỏ sót trường hợp độ dài lớn hơn 9 với lặp chữ số (ví dụ `n = 783`, xem mục 13).

In [11]:
def fascinating_set(n: int) -> bool:
    s = str(n) + str(2 * n) + str(3 * n)
    return len(s) == 9 and set(s) == set("123456789")

print(fascinating_set(192), fascinating_set(100), fascinating_set(267))

True False False


In [12]:
class SolutionSet:
    def isFascinating(self, n: int) -> bool:
        s = str(n) + str(2 * n) + str(3 * n)
        return len(s) == 9 and set(s) == set("123456789")

print(SolutionSet().isFascinating(192))

True


**Độ phức tạp:** tạo `set` `O(L)`, `L ≤ 11` ⇒ hằng số. Space `O(1)` (tập tối đa 10 phần tử).

### Cách 4 — Tách chữ số bằng số học, mảng đếm, thoát sớm

Không dùng chuỗi: với mỗi số trong `(n, 2n, 3n)`, lấy từng chữ số `d = x % 10`. Nếu `d == 0` hoặc `seen[d]` đã đánh dấu ⇒ **thất bại ngay** (thoát sớm); ngược lại đánh dấu. Sau khi xử lý cả ba số, ta phải đã thấy **đúng 9** chữ số khác nhau `1..9`.

Giải thích từng đoạn:

- `seen = [False] * 10`: `seen[d]` nói chữ số `d` đã thấy chưa.
- Vòng ngoài `for x in (n, 2 * n, 3 * n)`; vòng trong `while x > 0`: `x % 10` lấy chữ số, `x //= 10` bỏ chữ số.
- Gặp `0` hoặc gặp lại chữ số đã thấy ⇒ `return False`.
- Đếm `count` số chữ số hợp lệ; cuối cùng `count == 9` (khi đó vì không lặp và không có 0, đó chính là 9 chữ số `1..9`).

In [13]:
def fascinating_arith(n: int) -> bool:
    seen = [False] * 10
    count = 0
    for x in (n, 2 * n, 3 * n):
        while x > 0:
            d = x % 10
            if d == 0 or seen[d]:
                return False
            seen[d] = True
            count += 1
            x //= 10
    return count == 9

print(fascinating_arith(192), fascinating_arith(100), fascinating_arith(267), fascinating_arith(783))

True False False False


In [14]:
class SolutionArith:
    def isFascinating(self, n: int) -> bool:
        seen = [False] * 10
        count = 0
        for x in (n, 2 * n, 3 * n):
            while x > 0:
                d = x % 10
                if d == 0 or seen[d]:
                    return False
                seen[d] = True
                count += 1
                x //= 10
        return count == 9

print(SolutionArith().isFascinating(192))

True


**Độ phức tạp:** tối đa 11 chữ số xử lý, thực tế dừng sớm khi gặp `0` hoặc trùng ⇒ Time `O(1)` (hằng số), Space `O(1)` (mảng 10 ô).

Lưu ý: nếu `count < 9` ở cuối (ví dụ `n` nhỏ làm `3n` còn ít chữ số) thì `False` — nhưng với `n ≥ 100` luôn có ≥ 9 chữ số nên trường hợp này không xuất hiện; vẫn nên giữ điều kiện cho chắc.

## 8. TRACE từng bước

Trace Cách 4 (tách chữ số) với `n = 192`:

| số `x` | chữ số lấy ra (phải → trái) | `seen` cập nhật | `count` |
|---|---|---|---|
| `192` | `2`, `9`, `1` | đánh dấu 2, 9, 1 | 3 |
| `384` | `4`, `8`, `3` | đánh dấu 4, 8, 3 | 6 |
| `576` | `6`, `7`, `5` | đánh dấu 6, 7, 5 | 9 |

Kết quả `count == 9` và không gặp `0` hay trùng ⇒ `True`.

Với `n = 267`: `267`, `534`, `801`: lấy `1` rồi `0` ⇒ gặp `0` ⇒ `False` ngay. (Chuỗi `"267534801"` có 9 chữ số khác nhau nhưng chứa `0` và thiếu `9`.)

In [15]:
def trace_arith(n):
    seen = [False] * 10
    count = 0
    print(f"n = {n}: ba số {n}, {2 * n}, {3 * n}")
    for x in (n, 2 * n, 3 * n):
        print(f"  xử lý {x}:")
        while x > 0:
            d = x % 10
            if d == 0:
                print(f"    gặp chữ số 0 -> False")
                return False
            if seen[d]:
                print(f"    chữ số {d} đã thấy -> False")
                return False
            seen[d] = True
            count += 1
            print(f"    lấy {d}, count={count}")
            x //= 10
    print("  count =", count, "->", count == 9)
    return count == 9

trace_arith(192)

n = 192: ba số 192, 384, 576
  xử lý 192:
    lấy 2, count=1
    lấy 9, count=2
    lấy 1, count=3
  xử lý 384:
    lấy 4, count=4
    lấy 8, count=5
    lấy 3, count=6
  xử lý 576:
    lấy 6, count=7
    lấy 7, count=8
    lấy 5, count=9
  count = 9 -> True


True

In [16]:
# Trace thứ hai: thất bại vì chứa 0, và thất bại vì lặp chữ số
trace_arith(267)
trace_arith(111)

n = 267: ba số 267, 534, 801
  xử lý 267:
    lấy 7, count=1
    lấy 6, count=2
    lấy 2, count=3
  xử lý 534:
    lấy 4, count=4
    lấy 3, count=5
    lấy 5, count=6
  xử lý 801:
    lấy 1, count=7
    gặp chữ số 0 -> False
n = 111: ba số 111, 222, 333
  xử lý 111:
    lấy 1, count=1
    chữ số 1 đã thấy -> False


False

## 9. Code nộp LeetCode

Dùng Cách 2 (sắp xếp) hoặc Cách 3 (`set` + độ dài). Ta nộp Cách 2 vì ngắn và không thể quên điều kiện độ dài. Đúng class `Solution` và method `isFascinating`.

In [17]:
class Solution:
    def isFascinating(self, n: int) -> bool:
        s = str(n) + str(2 * n) + str(3 * n)
        return "".join(sorted(s)) == "123456789"

In [18]:
print(Solution().isFascinating(192))     # True
print(Solution().isFascinating(100))     # False

True
False


## 10. Vì sao thuật toán đúng

Gọi `s = str(n) + str(2n) + str(3n)`; mọi ký tự của `s` đều là chữ số `0..9`.

**(⇒) Nếu `n` fascinating thì `sorted(s) == "123456789"`.** Đề nói `s` chứa **mỗi chữ số `1..9` đúng một lần** và **không có `0`**. Vậy các ký tự của `s` đúng là chín ký tự `1,...,9`, mỗi cái một lần (độ dài 9, không còn chỗ cho ký tự nào khác). Sắp xếp tăng dần thu được `"123456789"`.

**(⇐) Nếu `sorted(s) == "123456789"` thì `n` fascinating.** Khi đó `s` có đúng 9 ký tự và chúng, sau khi sắp xếp, là `1,2,...,9`; tức mỗi chữ số `1..9` xuất hiện đúng một lần và `0` không xuất hiện. Đó chính là điều kiện của đề.

**Vì sao Cách 1 không cần kiểm tra độ dài.** `s` chỉ gồm các chữ số. Nếu `0` không có trong `s` và mỗi `1..9` xuất hiện **đúng một lần** thì tổng số ký tự là `9`; không còn ký tự nào khác. ⇒ Cách 1 (`"0"` không có + 9 lần `count == 1`) tương đương đề.

**Vì sao Cách 3 cần `len(s) == 9`.** Nếu `set(s) == {1,...,9}` mà `len(s) > 9` thì có chữ số lặp ⇒ vi phạm "đúng một lần". Cần cả hai điều kiện.

**Cách 4.** Thoát sớm khi gặp `0` hoặc chữ số lặp (vi phạm đề). Nếu không thoát và `count == 9` thì 9 chữ số đều khác nhau, không phải `0` ⇒ đúng 9 giá trị khác nhau trong `{1,...,9}` ⇒ mỗi giá trị một lần.

## 11. Phân tích Time / Space Complexity

Đầu vào `n` chỉ có 3 chữ số, chuỗi `s` dài `L ≤ 11`:

| Cách | Time | Space |
|---|---|---|
| 1. đếm 9 chữ số | `O(9·L)` = hằng số | `O(L)` |
| 2. sắp xếp + so sánh | `O(L log L)` = hằng số | `O(L)` |
| 3. `set` + độ dài | `O(L)` = hằng số | `O(1)` |
| 4. tách chữ số, thoát sớm | `O(L)` = hằng số | `O(1)` |

Về mặt lý thuyết: nếu `n` có `d` chữ số thì `L = O(d)`; Cách 2 là `O(d log d)`, còn các cách còn lại `O(d)`. Với `d = 3` mọi cách đều **chạy theo hằng số**, nên chọn cách **dễ viết đúng nhất**.

## 12. Bộ test

### 12.1. Ví dụ của đề

In [19]:
sols = [SolutionSorted(), SolutionSet(), SolutionArith(), Solution()]
funcs = [fascinating_count, fascinating_sorted, fascinating_set, fascinating_arith] + [s.isFascinating for s in sols]

for f in funcs:
    assert f(192) is True
    assert f(100) is False
print("Ví dụ của đề: tất cả cách đều đúng")

Ví dụ của đề: tất cả cách đều đúng


### 12.2. Edge case (chỉ dùng input hợp lệ: `100 <= n <= 999`)

- Hai đầu miền: `100` (`False`), `999` (`False`; `3n = 2997` dài 4 chữ số).
- Số lặp chữ số: `111`, `222`, `333`; chứa `0` trong `n`: `101`, `200`, `500`.
- Ngay ngoài ngưỡng `n ≤ 333`: `334` (`3n = 1002` → độ dài 10): `False`.
- Bốn số thoả (kết quả vét cạn bên dưới): `192, 219, 273, 327`.
- Gần đúng nhưng sai: `267` (`"267534801"`: có `0`, thiếu `9`), `783` (set đủ nhưng độ dài 11).

In [20]:
cases = {
    100: False, 999: False, 111: False, 222: False, 333: False, 101: False, 200: False, 500: False,
    334: False, 267: False, 783: False,
    192: True, 219: True, 273: True, 327: True,
}
for f in funcs:
    for n, want in cases.items():
        assert f(n) is want, (f, n, want)
print("Edge case OK:", len(cases), "trường hợp x", len(funcs), "cách")

Edge case OK: 15 trường hợp x 8 cách


### 12.3. Phản ví dụ cho cách làm sai

Các cách sai hay gặp (chi tiết ở mục 13): **cộng số** thay vì nối chuỗi, chỉ kiểm tra `set` mà thiếu độ dài, và so sánh `sorted(s)` (list) với chuỗi. Ta xác nhận mỗi cách sai ở ít nhất một `n`.

In [21]:
def sai_cong_so(n):
    s = str(n + 2 * n + 3 * n)                # cộng số = 6n
    return "".join(sorted(s)) == "123456789"

def sai_chi_set(n):
    s = str(n) + str(2 * n) + str(3 * n)
    return set(s) == set("123456789")

def sai_sorted_list(n):
    s = str(n) + str(2 * n) + str(3 * n)
    return sorted(s) == "123456789"

print("cộng số 192:", sai_cong_so(192), "(đúng True)")
print("chỉ set 783:", sai_chi_set(783), "(đúng False)")
print("sorted(list) 192:", sai_sorted_list(192), "(đúng True)")
assert sai_cong_so(192) is False
assert sai_chi_set(783) is True
assert sai_sorted_list(192) is False

cộng số 192: False (đúng True)
chỉ set 783: True (đúng False)
sorted(list) 192: False (đúng True)


### 12.4. Vét cạn toàn miền `100..999` + so sánh các cách

Miền hợp lệ chỉ có **900** giá trị nên ta có thể kiểm tra **tất cả**. Oracle độc lập: dùng `collections.Counter` đếm số lần mỗi ký tự của `s` và so với `Counter("123456789")`. Sau đó rút ra danh sách các `n` thoả và kiểm tra chúng đúng là `[192, 219, 273, 327]`.

In [22]:
from collections import Counter

def oracle(n):
    s = str(n) + str(2 * n) + str(3 * n)
    return Counter(s) == Counter("123456789")

good = []
for n in range(100, 1000):
    want = oracle(n)
    for f in funcs:
        assert f(n) is want, (n, f)
    if want:
        good.append(n)
print("các số fascinating trong [100, 999]:", good)
assert good == [192, 219, 273, 327]
print("OK: đã kiểm toàn bộ 900 giá trị với", len(funcs), "cách")

các số fascinating trong [100, 999]: [192, 219, 273, 327]
OK: đã kiểm toàn bộ 900 giá trị với 8 cách


Ghi chú: ở miền 3 chữ số chỉ có đúng 4 số fascinating, nhưng ta **không dùng bảng tra** `n in {192, 219, 273, 327}` làm lời giải chính, vì (1) nó chỉ đúng cho **đúng đề này** và sẽ sai ngay khi đề thay đổi (số chữ số khác, nhân với `2, 3, 4`...), còn cách kiểm tra tổng quát thì vẫn dùng được; (2) mục đích học là hiểu **cách kiểm tra**; bảng tra chỉ là hệ quả của vét cạn. Xem biến thể ở mục 15.

## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cộng số thay vì nối chuỗi

In [23]:
def sai_cong(n: int) -> bool:
    total = n + 2 * n + 3 * n               # 6n: đây là CỘNG, không phải nối
    return "".join(sorted(str(total))) == "123456789"

print("sai :", sai_cong(192))               # False (192 + 384 + 576 = 1152)
print("đúng:", Solution().isFascinating(192))   # True
assert sai_cong(192) is False

sai : False
đúng: True


**Vì sao sai:** đề nói "**nối**" (concatenate): `192`, `384`, `576` ⇒ `"192384576"`. Cộng ra `1152` không còn ý nghĩa. Luôn `str(...)` từng số **trước** khi `+`.

### Lỗi 2: Chỉ kiểm tra `set`, quên độ dài

In [24]:
def sai_chi_set_2(n: int) -> bool:
    s = str(n) + str(2 * n) + str(3 * n)
    return set(s) == set("123456789")       # SAI: thiếu len(s) == 9

print("n = 783 ->", str(783) + str(2 * 783) + str(3 * 783))
print("sai :", sai_chi_set_2(783))          # True nhưng sai
print("đúng:", Solution().isFascinating(783))   # False
assert sai_chi_set_2(783) is True and Solution().isFascinating(783) is False

n = 783 -> 78315662349
sai : True
đúng: False


**Vì sao sai:** chuỗi của `n = 783` là `"78315662349"` (`783`, `1566`, `2349`, độ dài 11) chứa đủ các chữ số `1..9` nhưng **lặp** chữ số; `set` quên mất tần suất. Điều kiện "**đúng một lần**" cần thêm `len(s) == 9` (hoặc dùng sắp xếp/đếm).

### Lỗi 3: Chỉ kiểm tra độ dài 9 và 9 chữ số khác nhau, quên loại `0`

In [25]:
def sai_quen_loai_0(n: int) -> bool:
    s = str(n) + str(2 * n) + str(3 * n)
    return len(s) == 9 and len(set(s)) == 9      # SAI: chưa chắc là 1..9 (có thể chứa 0)

print("n = 267 ->", str(267) + str(2 * 267) + str(3 * 267))     # 267534801
print("sai :", sai_quen_loai_0(267))        # True
print("đúng:", Solution().isFascinating(267))   # False
assert sai_quen_loai_0(267) is True and Solution().isFascinating(267) is False

n = 267 -> 267534801
sai : True
đúng: False


**Vì sao sai:** `"267534801"` có 9 chữ số khác nhau nhưng thuộc tập `{0,1,2,3,4,5,6,7,8}`: chứa `0` và **thiếu `9`**. Đề yêu cầu đúng tập `{1,...,9}`, **không có `0`**. So với tập `set("123456789")` hoặc dùng sắp xếp để tránh quên.

### Lỗi 4: So sánh `sorted(s)` (list) với chuỗi

In [26]:
def sai_so_list_voi_str(n: int) -> bool:
    s = str(n) + str(2 * n) + str(3 * n)
    return sorted(s) == "123456789"             # SAI: list so với str luôn khác nhau

print(sorted("192384576") == "123456789")       # False
print(sorted("192384576") == list("123456789")) # True
print("sai :", sai_so_list_voi_str(192))        # False
print("đúng:", Solution().isFascinating(192))   # True
assert sai_so_list_voi_str(192) is False

False
True
sai : False
đúng: True


**Vì sao sai:** `sorted(...)` luôn trả về **list**, mà `["1","2",...] == "123456789"` luôn `False` (kiểu khác nhau). Phải `"".join(sorted(s))` hoặc so với `list("123456789")`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chứa mỗi chữ số/ký tự **đúng một lần**" ⇒ so **sắp xếp** với dãy chuẩn, hoặc `Counter`, hoặc `len == k` **và** `set == tập chuẩn`.
- "**Nối**" ⇒ chuyển sang chuỗi bằng `str(...)` rồi `+`; nếu cần nối nhiều số: `"".join(map(str, [n, 2*n, 3*n]))`.
- Miền đầu vào **rất nhỏ** (ở đây 900 giá trị) ⇒ có thể **vét cạn toàn miền** để kiểm tra chắc chắn mọi cách giải.
- Hãy nghĩ **điều kiện cần** nhanh: tổng số chữ số phải là 9 ⇒ `n ≤ 333`; có thể trả `False` ngay nếu `n > 333` (tối ưu nhỏ, nhưng không bắt buộc).
- Khi `Counter`/`sorted` đủ rõ ràng thì không cần tối ưu thêm — bài này đầu vào hằng số.
- Cẩn thận kiểu dữ liệu: list ≠ chuỗi, số ≠ chuỗi.

## 15. Biến thể của bài

1. **Số `d` chữ số bất kỳ**, nối `n, 2n, 3n` rồi kiểm tra "pandigital" (đủ mọi chữ số `1..9` đúng một lần) — chỉ có nghiệm nếu độ dài tổng bằng 9.
2. **Pandigital tổng quát** (ví dụ "chuỗi có mọi chữ số `0..9` đúng một lần"): so sánh với `"0123456789"`.
3. **Nối `n, 2n, ..., kn`** (hằng số Champernowne / "concatenated product" trong Project Euler 38): tìm lớn nhất pandigital.
4. **Liệt kê tất cả số fascinating**: vét cạn như mục 12.4: `[192, 219, 273, 327]`.
5. **Bảng tra** `n in {192, 219, 273, 327}`: nhanh nhưng "mù" về nguyên nhân; chỉ nên dùng sau khi hiểu và kiểm chứng.
6. **Đếm bằng bitmask**: `mask |= 1 << d`; fascinating khi `mask == 0b1111111110` và đủ 9 chữ số.

In [27]:
def all_fascinating(lo=100, hi=999):
    return [n for n in range(lo, hi + 1) if Solution().isFascinating(n)]

print(all_fascinating())                       # [192, 219, 273, 327]

def fascinating_mask(n):
    mask, count = 0, 0
    for x in (n, 2 * n, 3 * n):
        while x > 0:
            mask |= 1 << (x % 10)
            count += 1
            x //= 10
    return count == 9 and mask == 0b1111111110

print([n for n in range(100, 1000) if fascinating_mask(n)])      # [192, 219, 273, 327]
print(fascinating_mask(192), fascinating_mask(267))             # True False

[192, 219, 273, 327]
[192, 219, 273, 327]
True False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. đếm từng chữ số | `"0" not in s` và `count == 1` cho `1..9` | `O(9·L)` | `O(L)` | dịch nguyên văn đề, dễ kiểm tra |
| 2. sắp xếp rồi so sánh | `"".join(sorted(s)) == "123456789"` | `O(L log L)` | `O(L)` | **cách nộp bài khuyến nghị**: ngắn, khó sai |
| 3. `set` + độ dài | `len(s) == 9 and set(s) == {1..9}` | `O(L)` | `O(1)` | khi muốn tránh sắp xếp |
| 4. tách chữ số bằng số học | `% 10`, `// 10`, mảng đánh dấu, thoát sớm | `O(L)` | `O(1)` | không dùng chuỗi; thoát sớm khi sai |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"12" + "3"`, `12 + 3`, `"".join(sorted("3124"))`, `set("1223")`.

**Bài 2 (tính tay).** Với `n = 219`: tính `2n`, `3n`, chuỗi nối, và cho biết có fascinating không.

**Bài 3 (điền TODO).** Hoàn thành `fascinating_todo(n)` bằng cách sắp xếp.

In [28]:
def fascinating_todo(n):
    # TODO 1: s = str(n) + str(2 * n) + str(3 * n)
    # TODO 2: so sánh "".join(sorted(s)) với "123456789"
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `fascinating_set2(n)` bằng `set` và độ dài mà không nhìn Cách 3. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `is_pandigital_0_9(s)` kiểm tra chuỗi `s` có đủ **mười** chữ số `0..9` mỗi chữ số đúng một lần. (b) Viết `concatenated_product(n, k)` trả chuỗi nối `n, 2n, ..., kn`; rồi `is_pandigital_product(n, k)` kiểm tra nó là một hoán vị của `1..9` (Project Euler 38) và tìm `n` **lớn nhất** thuộc `[1, 9999]` thoả với `k = 2`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"123"`; `15`; `"1234"`; `{"1","2","3"}`. Bài 2: `2n = 438`, `3n = 657`.
- **Mức 2 (vừa):** Bài 3: `s = ...` rồi `return "".join(sorted(s)) == "123456789"`. Bài 4: `len(s) == 9 and set(s) == set("123456789")`.
- **Mức 3 (sâu):** Bài 5(a): `"".join(sorted(s)) == "0123456789"`. Bài 5(b): `concatenated_product`: `"".join(str(n * i) for i in range(1, k + 1))`; duyệt `n` từ `9999` xuống `1` với `k = 2` rồi dừng ở nghiệm đầu tiên (Project Euler 38 cho đáp án `9327` với `k = 2` ra `932718654`).

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
assert "12" + "3" == "123"
assert 12 + 3 == 15
assert "".join(sorted("3124")) == "1234"
assert set("1223") == {"1", "2", "3"}

# Bài 2: n = 219 -> 438, 657 -> "219438657": đủ 1..9, không có 0
n = 219
assert (2 * n, 3 * n) == (438, 657)
s = str(n) + str(2 * n) + str(3 * n)
assert s == "219438657" and "".join(sorted(s)) == "123456789"
assert Solution().isFascinating(219) is True

# Bài 3
def fascinating_todo_ans(n):
    s = str(n) + str(2 * n) + str(3 * n)
    return "".join(sorted(s)) == "123456789"

assert fascinating_todo_ans(192) is True
assert fascinating_todo_ans(100) is False

# Bài 4
def fascinating_set2(n):
    s = str(n) + str(2 * n) + str(3 * n)
    return len(s) == 9 and set(s) == set("123456789")

assert fascinating_set2(192) is True
assert fascinating_set2(100) is False

# Bài 5
def is_pandigital_0_9(s):
    return "".join(sorted(s)) == "0123456789"

assert is_pandigital_0_9("1234567890") is True
assert is_pandigital_0_9("123456789") is False

def concatenated_product(n, k):
    return "".join(str(n * i) for i in range(1, k + 1))

def is_pandigital_product(n, k):
    return "".join(sorted(concatenated_product(n, k))) == "123456789"

assert concatenated_product(192, 3) == "192384576"
assert is_pandigital_product(192, 3) is True
best = next(n for n in range(9999, 0, -1) if is_pandigital_product(n, 2))
assert (best, concatenated_product(best, 2)) == (9327, "932718654")
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nối chuỗi → sắp xếp → so với `"123456789"`)

```python
class Solution:
    def isFascinating(self, n: int) -> bool:
        s = str(n) + str(2 * n) + str(3 * n)
        return "".join(sorted(s)) == "123456789"
```

**Checklist**

- [ ] **Nối chuỗi** (`str(...) + str(...)`), không **cộng số**?
- [ ] Có đủ cả hai điều kiện: đủ `1..9` đúng một lần **và** không có `0`? (sắp xếp so với `"123456789"` gói cả hai)
- [ ] Nếu dùng `set` thì có kèm `len(s) == 9`?
- [ ] Không so sánh `list` (`sorted(...)`) với `str` mà không `join`?
- [ ] Đã thử `192` (đúng), `100` (có 0), `267` (9 chữ số khác nhau nhưng có 0), `783` (set đủ nhưng dài 11)?
- [ ] Biết miền chỉ có 900 giá trị nên có thể vét cạn kiểm tra?
- [ ] Signature đúng `isFascinating(self, n: int) -> bool`?